# 01 Data Setup And Train GAN

Use this notebook for dataset sanity checks, image previews, and a standalone GAN run in `runs/gan/`. The shared pipeline now targets a local CPU-only setup, so notebook 01 stays simple and laptop-friendly.


In [ ]:
from pathlib import Path
import os
import sys
from pprint import pprint

from IPython.display import display
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from config import Config
from pipeline import sample_image_paths, summarize_dataset, train_gan

ROOT


In [ ]:
# Edit these values before running.
cfg = Config().with_runtime_profile_defaults()
DATA_ROOT = ROOT / "data_final"
OUT_DIR = ROOT / "runs" / "gan"
TRAIN_N_PER_CLASS = None
STRICT_FID = True
NUM_WORKERS_OVERRIDE = None

# Optional quick smoke-run overrides for notebook debugging.
SMOKE_GAN_OVERRIDES = {}
overrides = dict(SMOKE_GAN_OVERRIDES)
if NUM_WORKERS_OVERRIDE is not None:
    overrides["num_workers"] = NUM_WORKERS_OVERRIDE
if overrides:
    cfg = cfg.with_overrides(**overrides)

{
    "cfg": cfg,
    "device": "cpu",
    "loader_options": cfg.loader_options(),
    "data_root": DATA_ROOT,
    "out_dir": OUT_DIR,
    "train_n_per_class": TRAIN_N_PER_CLASS,
    "strict_fid": STRICT_FID,
}


In [ ]:
dataset_summary = summarize_dataset(DATA_ROOT)
pprint(dataset_summary)

for class_name, sample_path in sample_image_paths(DATA_ROOT):
    print(class_name, sample_path.name)
    display(Image.open(sample_path))


## GAN Training

This notebook keeps the standalone GAN workflow available. For the full Task 1 pipeline, use `02_generate_synthetic_data_and_classifier_experiments.ipynb`.


In [ ]:
gan_run = train_gan(
    cfg,
    data_root=DATA_ROOT,
    fid_root=DATA_ROOT,
    out_dir=OUT_DIR,
    train_n_per_class=TRAIN_N_PER_CLASS,
    strict_fid=STRICT_FID,
)
gan_run["summary"]
